### Investigation into High-Frequency Features and Targets

https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4095405


In [2]:
! pip install ruptures

import ruptures as rpt  # our package
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.api import VAR, ccf
import sys
import os
import logging

from datetime import datetime, timezone, timedelta
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from collections import deque


sys.path.append("/home/jovyan/data_warehouse/")
from dw_src.api.features_api import get_features_df, load_features

  Using cached ruptures-1.1.7-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.1 MB)


# Generate Features 

### Book Features
- Bid-Ask Spread (Variations)
- LOB Imbalance
- Midprice Returns

### Trade Features
- Roll Measure ( covariance of returns )
- Order Size Volume (Variations)
- Kyle's Lambda  (Variations)
- Order Size Imbalance
- Order Side Imbalance
- VPIN /PIN

### Targets
- Bid-Ask Spread (Variations)
- Returns (Sign and Size)
- Transaction Duration

In [9]:
# Choose some features
features_to_load = [
    "spread_mean",
    "midprice_mean",
    "bbo_imbalance_mean",
    'trade_side_mean',# order side imbalance
    'trade_count',
    "Volume", # dollar volume
]

# Last 2 weeks
end = datetime.strptime("2022-6-2","%Y-%m-%d")
start= datetime.strptime("2022-6-1","%Y-%m-%d")

# Choose buckketing
time_agg = "5sec" # 1sec/5sec/10sec/30sec/60sec

df_data = load_features(
    "ftx", "BTCUSDP", time_agg, start, end, features_to_load
)
df_data.columns=[str(i+"_5") for i in df_data.columns]
df_data['Returns_5']=np.zeros(df_data.shape[0]) # many of these are zero as price doesn't always shift -> leads to illdefined correlations 
df_data['Returns_5']=pd.Series(df_data['midprice_mean_5'].pct_change(),index=df_data.index[1:])*100
df_data['Lambda_5']=df_data['Returns_5']/df_data['Volume_5'] # 5 second lookback which is our default here 

  
df_data['Roll_30']=2*np.sqrt(df_data['Returns_5'].rolling(6).cov(df_data['Returns_5'].shift(1).rolling(6)))
df_data

/opt/conda/lib/python3.8/site-packages/google/auth/_default.py:70: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. We recommend you rerun `gcloud auth application-default login` and make sure a quota project is added. Or you can use service accounts instead. For more information about service accounts, see https://cloud.google.com/docs/authentication/
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
Downloading: 100%|██████████| 17162/17162 [00:01<00:00, 8946.90rows/s]
/opt/conda/lib/python3.8/site-packages/pandas/core/series.py:726: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


,exchange_5,symbol_5,spread_mean_5,midprice_mean_5,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30
timestamp,,,,,,,,,,,
2022-06-01 00:00:00+00:00,ftx,BTCUSDP,1.041322,31782.446281,0.834091,0.250000,16,4.9002,NaN,NaN,NaN
2022-06-01 00:00:05+00:00,ftx,BTCUSDP,1.000000,31785.500000,0.651726,0.363636,11,2.1804,0.009608,0.004407,NaN
2022-06-01 00:00:10+00:00,ftx,BTCUSDP,1.000000,31785.500000,0.778515,0.625000,8,0.3668,0.000000,0.000000,NaN
2022-06-01 00:00:15+00:00,ftx,BTCUSDP,1.035294,31791.282353,0.838995,0.095238,21,1.8892,0.018192,0.009629,NaN
2022-06-01 00:00:20+00:00,ftx,BTCUSDP,1.000000,31793.500000,0.740781,0.444444,9,0.7860,0.006976,0.008875,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2022-06-01 23:59:35+00:00,ftx,BTCUSDP,1.000000,29773.500000,0.522203,0.100000,10,0.5260,0.000000,0.000000,NaN
2022-06-01 23:59:40+00:00,ftx,BTCUSDP,1.000000,29773.500000,0.435986,0.500000,14,1.3337,0.000000,0.000000,0.005256
2022-06-01 23:59:45+00:00,ftx,BTCUSDP,1.048780,29778.117886,0.415356,0.063830,47,15.3064,0.015510,0.001013,NaN


### Generate Targets -> 5/30 Seconds
Transaction Return 
- The average return of midprices over a future window

Price Direction
- Binary variable relating future window of returns to current (avg)

Transaction Duration (TO:DO)
- Amount of time required for some value of transactions/volume etc. to transact 


In [10]:
df_data['Target_Return_5']=df_data['Returns_5'].shift(-1)
df_data['Target_Return_30']=df_data['Returns_5'].shift(-6)

df_data['Target_Price_Dir5']=(df_data['Target_Return_5']>df_data['Returns_5'].mean()).astype(int)

df_data['Target_Price_Dir30']=(df_data['Target_Return_30']>df_data['Returns_5'].mean()).astype(int)

In [11]:
df_data

,exchange_5,symbol_5,spread_mean_5,midprice_mean_5,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30,Target_Return_5,Target_Return_30,Target_Price_Dir5,Target_Price_Dir30
timestamp,,,,,,,,,,,,,,,
2022-06-01 00:00:00+00:00,ftx,BTCUSDP,1.041322,31782.446281,0.834091,0.250000,16,4.9002,NaN,NaN,NaN,0.009608,0.000000,1,1
2022-06-01 00:00:05+00:00,ftx,BTCUSDP,1.000000,31785.500000,0.651726,0.363636,11,2.1804,0.009608,0.004407,NaN,0.000000,-0.014908,1,0
2022-06-01 00:00:10+00:00,ftx,BTCUSDP,1.000000,31785.500000,0.778515,0.625000,8,0.3668,0.000000,0.000000,NaN,0.018192,-0.007110,1,0
2022-06-01 00:00:15+00:00,ftx,BTCUSDP,1.035294,31791.282353,0.838995,0.095238,21,1.8892,0.018192,0.009629,NaN,0.006976,0.015730,1,1
2022-06-01 00:00:20+00:00,ftx,BTCUSDP,1.000000,31793.500000,0.740781,0.444444,9,0.7860,0.006976,0.008875,NaN,0.000000,0.028130,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-06-01 23:59:35+00:00,ftx,BTCUSDP,1.000000,29773.500000,0.522203,0.100000,10,0.5260,0.000000,0.000000,NaN,0.000000,NaN,1,0
2022-06-01 23:59:40+00:00,ftx,BTCUSDP,1.000000,29773.500000,0.435986,0.500000,14,1.3337,0.000000,0.000000,0.005256,0.015510,NaN,1,0
2022-06-01 23:59:45+00:00,ftx,BTCUSDP,1.048780,29778.117886,0.415356,0.063830,47,15.3064,0.015510,0.001013,NaN,0.011358,NaN,1,0


# Least Squares Relationship between Features and Targets

### Mean Return 5 seconds into Future

In [119]:
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

coeff_df=pd.DataFrame()

In [120]:
features= [ "bbo_imbalance_mean_5","trade_side_mean_5","trade_count_5","Volume_5","Returns_5","Lambda_5","Roll_30"] # All are 5 second lookback except for Roll -> Requires correlation on window 
targets=["Target_Return_5","Target_Return_30","Target_Price_Dir5","Target_Price_Dir30"]

In [121]:
df_data=df_data.dropna()
X_train, X_test, y_train, y_test = train_test_split(df_data[features],df_data['Target_Return_5'])

In [122]:
X_train

,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30
timestamp,,,,,,,
2022-06-01 01:08:40+00:00,0.910373,0.000000,1,0.0040,0.000000,0.000000,0.011689
2022-06-01 16:58:45+00:00,0.201100,0.566667,30,12.6060,-0.001712,-0.000136,0.008712
2022-06-01 20:33:15+00:00,0.416985,0.382488,217,79.3252,0.096358,0.001215,0.110213
2022-06-01 07:01:45+00:00,0.588020,0.000000,1,0.0022,-0.000457,-0.207583,0.023123
2022-06-01 07:16:15+00:00,0.921303,0.000000,8,1.5170,0.016550,0.010909,0.005600
...,...,...,...,...,...,...,...
2022-06-01 07:49:15+00:00,0.182120,0.052632,57,29.4343,-0.010905,-0.000370,0.040589
2022-06-01 04:01:25+00:00,0.677525,0.011765,85,13.7338,0.019587,0.001426,0.006145
2022-06-01 20:20:00+00:00,0.514826,0.405405,37,6.7424,-0.017096,-0.002536,0.022512


Let's normalize our data s.t. no individual feature overflows /skews the penalty 

This is very important in LASSO, otherwise it is possible for a noniformative feature to "drown" out informative features soley on magnitudes 

In [123]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train=pd.DataFrame(scaler.transform(X_train))

scaler.fit(y_train.values.reshape(-1, 1))
y_train=scaler.transform(y_train.values.reshape(-1, 1))

In [124]:
model= linear_model.Lasso(0.01) # here use an alpha penalty of 0.01
model.fit(X_train,y_train)

Lasso(alpha=0.01)

In [125]:
coeffs=np.array(model.coef_) # optimal coefficients
coeff_df[0]=coeffs
for i,c in enumerate(coeffs):
    print(f"{features[i]} best estimate is {c} ")

bbo_imbalance_mean_5 best estimate is 0.11767636526880995 
trade_side_mean_5 best estimate is -0.22256874304808005 
trade_count_5 best estimate is -0.00760977961605148 
Volume_5 best estimate is -0.025224662070313673 
Returns_5 best estimate is 0.15456189595960715 
Lambda_5 best estimate is -0.0 
Roll_30 best estimate is -0.0 


### Mean Return 30 Seconds into Future

In [126]:
df_data=df_data.dropna()
X_train, X_test, y_train, y_test = train_test_split(df_data[features],df_data['Target_Return_30'])

In [127]:
X_train

,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30
timestamp,,,,,,,
2022-06-01 08:13:45+00:00,0.376208,1.000000,1,0.0524,-0.000040,-0.000757,0.006732
2022-06-01 10:12:05+00:00,0.316389,0.375000,8,0.6450,-0.005549,-0.008604,0.009777
2022-06-01 06:38:00+00:00,0.686373,0.000000,37,19.2161,0.036462,0.001897,0.020728
2022-06-01 02:12:25+00:00,0.251241,0.833333,6,0.1678,-0.014659,-0.087359,0.009619
2022-06-01 21:21:35+00:00,0.665036,0.272727,11,2.8874,-0.000289,-0.000100,0.008485
...,...,...,...,...,...,...,...
2022-06-01 08:37:05+00:00,0.881086,0.666667,3,0.2286,0.001483,0.006487,0.007405
2022-06-01 21:49:55+00:00,0.894053,0.500000,6,1.3060,0.003090,0.002366,0.003010
2022-06-01 10:34:35+00:00,0.219703,1.000000,7,0.2100,0.000000,0.000000,0.007581


In [128]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train=pd.DataFrame(scaler.transform(X_train))

scaler.fit(y_train.values.reshape(-1, 1))
y_train=scaler.transform(y_train.values.reshape(-1, 1))

In [129]:
model= linear_model.Lasso(0.01) # here use an alpha penalty of 0.01
model.fit(X_train,y_train)

Lasso(alpha=0.01)

In [130]:
coeffs=np.array(model.coef_) # optimal coefficients
coeff_df[1]=coeffs
for i,c in enumerate(coeffs):
    print(f"{features[i]} best estimate is {c} ")

bbo_imbalance_mean_5 best estimate is 0.01130655686862369 
trade_side_mean_5 best estimate is -0.0 
trade_count_5 best estimate is -0.0 
Volume_5 best estimate is -0.019618774013093056 
Returns_5 best estimate is -0.0 
Lambda_5 best estimate is 0.0 
Roll_30 best estimate is 0.007663721801514918 


### Binary Price Direction of Mean Return 5 Seconds into Future

In [131]:
df_data=df_data.dropna()
X_train, X_test, y_train, y_test = train_test_split(df_data[features],df_data['Target_Price_Dir5'])

In [132]:
X_train

,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30
timestamp,,,,,,,
2022-06-01 11:38:25+00:00,0.482193,1.000000,1,0.1600,0.000000,0.000000,0.018527
2022-06-01 01:35:05+00:00,0.227595,0.913043,23,1.9036,-0.043372,-0.022784,0.033044
2022-06-01 08:07:40+00:00,0.452124,0.062500,16,3.1560,0.013440,0.004259,0.018267
2022-06-01 04:45:10+00:00,0.352415,0.027027,74,14.5559,0.051685,0.003551,0.008232
2022-06-01 02:23:20+00:00,0.771558,0.200000,30,4.7318,0.019630,0.004149,0.017779
...,...,...,...,...,...,...,...
2022-06-01 21:41:50+00:00,0.822720,0.000000,16,1.1018,0.001307,0.001186,0.012709
2022-06-01 00:13:45+00:00,0.359244,0.250000,8,2.6937,0.013535,0.005025,0.023984
2022-06-01 10:43:10+00:00,0.975342,0.000000,1,0.0015,0.000000,0.000000,0.000000


In [133]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train=pd.DataFrame(scaler.transform(X_train))

scaler.fit(y_train.values.reshape(-1, 1))
y_train=scaler.transform(y_train.values.reshape(-1, 1))

In [134]:
model= linear_model.Lasso(0.01) # here use an alpha penalty of 0.01
model.fit(X_train,y_train)

Lasso(alpha=0.01)

In [135]:
coeffs=np.array(model.coef_) # optimal coefficients
coeff_df[2]=coeffs
for i,c in enumerate(coeffs):
    print(f"{features[i]} best estimate is {c} ")

bbo_imbalance_mean_5 best estimate is 0.20244835207021725 
trade_side_mean_5 best estimate is -0.288295963697359 
trade_count_5 best estimate is -0.08730852487717815 
Volume_5 best estimate is 0.02304014405227852 
Returns_5 best estimate is 0.0337934072185741 
Lambda_5 best estimate is -0.0 
Roll_30 best estimate is -0.06276051451739058 


### Binary Price Direction of Mean Return 30 Seconds into Future 

In [136]:
df_data=df_data.dropna()
X_train, X_test, y_train, y_test = train_test_split(df_data[features],df_data['Target_Price_Dir30'])

In [137]:
X_train

,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30
timestamp,,,,,,,
2022-06-01 19:29:35+00:00,0.869609,0.500000,4,0.0689,0.011665,0.169297,0.029539
2022-06-01 11:51:45+00:00,0.803327,0.000000,3,0.0869,0.000000,0.000000,0.006858
2022-06-01 21:56:15+00:00,0.559597,0.807692,26,3.7240,0.025832,0.006937,0.018752
2022-06-01 00:46:00+00:00,0.268507,0.956522,23,7.2092,-0.013227,-0.001835,0.022003
2022-06-01 11:29:25+00:00,0.158954,0.933333,15,1.0191,-0.006059,-0.005946,0.011453
...,...,...,...,...,...,...,...
2022-06-01 05:59:15+00:00,0.714693,0.000000,18,4.4793,0.000435,0.000097,0.011215
2022-06-01 14:50:55+00:00,0.146935,0.706667,75,15.5222,-0.023237,-0.001497,0.025203
2022-06-01 14:53:45+00:00,0.781113,0.767442,43,12.1202,-0.025258,-0.002084,0.008656


In [138]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train=pd.DataFrame(scaler.transform(X_train))

scaler.fit(y_train.values.reshape(-1, 1))
y_train=scaler.transform(y_train.values.reshape(-1, 1))

In [139]:
model= linear_model.Lasso(0.01) # here use an alpha penalty of 0.01
model.fit(X_train,y_train)

Lasso(alpha=0.01)

In [140]:
coeffs=np.array(model.coef_) # optimal coefficients
coeff_df[3]=coeffs
for i,c in enumerate(coeffs):
    print(f"{features[i]} best estimate is {c} ")

bbo_imbalance_mean_5 best estimate is 0.019603967091464548 
trade_side_mean_5 best estimate is -0.0 
trade_count_5 best estimate is -0.06758995213941839 
Volume_5 best estimate is -0.0 
Returns_5 best estimate is -0.0 
Lambda_5 best estimate is 0.0 
Roll_30 best estimate is -0.04163311286026662 


# Results 

### LASSO $\alpha=0.01$

In [141]:
coeff_df=coeff_df.T
coeff_df.columns=features
coeff_df.index=targets

### Set a 0.05 confidence 
As our features are standardized this does not have any undesired effect on scaling properties of features

In [144]:
sig_coeff_df=coeff_df[abs(coeff_df)>0.05].fillna(0)
sig_coeff_df

,bbo_imbalance_mean_5,trade_side_mean_5,trade_count_5,Volume_5,Returns_5,Lambda_5,Roll_30
Target_Return_5,0.117676,-0.222569,0.000000,0.0,0.154562,0.0,0.000000
Target_Return_30,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000
Target_Price_Dir5,0.202448,-0.288296,-0.087309,0.0,0.000000,0.0,-0.062761
Target_Price_Dir30,0.000000,0.000000,-0.067590,0.0,0.000000,0.0,0.000000


### Conclusions 
    -  BBO Imbalance Mean, Latest Returns are positively correlated with returns 5 seconds in future
    - Trade Side Mean is reverting on returns 5 seconds in future
    - No Features Informative for 30 Second Lookahead 